# **Problem Statement**

## Business Context

A sales forecast is a prediction of future sales revenue based on historical data, industry trends, and the status of the current sales pipeline. Businesses use the sales forecast to estimate weekly, monthly, quarterly, and annual sales totals. A company needs to make an accurate sales forecast as it adds value across an organization and helps the different verticals to chalk out their future course of action.

Forecasting helps an organization plan its sales operations by region and provides valuable insights to the supply chain team regarding the procurement of goods and materials. An accurate sales forecast process has many benefits which include improved decision-making about the future and reduction of sales pipeline and forecast risks. Moreover, it helps to reduce the time spent in planning territory coverage and establish benchmarks that can be used to assess trends in the future.

## Objective

SuperKart is a retail chain operating supermarkets and food marts across various tier cities, offering a wide range of products. To optimize its inventory management and make informed decisions around regional sales strategies, SuperKart wants to accurately forecast the sales revenue of its outlets for the upcoming quarter.

To operationalize these insights at scale, the company has partnered with a data science firm—not just to build a predictive model based on historical sales data, but to develop and deploy a robust forecasting solution that can be integrated into SuperKart's decision-making systems and used across its network of stores.

## Data Description

The data contains the different attributes of the various products and stores. The detailed data dictionary is given below.

- **Product_Id** - unique identifier of each product
- **Product_Weight** - weight of each product
- **Product_Sugar_Content** - sugar content: low sugar, regular, no sugar
- **Product_Allocated_Area** - ratio of allocated display area to total display area in a store
- **Product_Type** - broad category: meat, snack foods, hard drinks, dairy, etc.
- **Product_MRP** - maximum retail price of each product
- **Store_Id** - unique identifier of each store
- **Store_Establishment_Year** - year in which the store was established
- **Store_Size** - size: high, medium, low
- **Store_Location_City_Type** - Tier 1, Tier 2, or Tier 3
- **Store_Type** - Departmental Store, Supermarket Type1, Supermarket Type2, Food Mart
- **Product_Store_Sales_Total** - total revenue generated by the product in that store

# **Installing and Importing the necessary libraries**

In [ ]:
#Installing the libraries with the specified versions
!pip install numpy==2.0.2 pandas==2.2.2 scikit-learn==1.6.1 matplotlib==3.10.0 seaborn==0.13.2 joblib==1.4.2 xgboost==2.1.4 requests==2.32.4 huggingface_hub==0.34.0 -q

**Note:** After running the above cell, restart the kernel and run all cells sequentially from the next cell.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

from sklearn.ensemble import (
    BaggingRegressor, RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
)
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor

from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
)

from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder

import joblib
import os
import requests
from huggingface_hub import login, HfApi, create_repo

print('Libraries imported successfully.')

# **Loading the dataset**

In [ ]:
kart = pd.read_csv('SuperKart.csv')
data = kart.copy()
print(f'Dataset loaded: {data.shape[0]} rows, {data.shape[1]} columns')

# **Data Overview**

In [ ]:
# First 5 rows
data.head()

In [ ]:
# Last 5 rows
data.tail()

In [ ]:
print(f'There are {data.shape[0]} rows and {data.shape[1]} columns.')

In [ ]:
data.info()

In [ ]:
# Statistical summary
data.describe(include='all').T

In [ ]:
# Check duplicates
print('Duplicate rows:', data.duplicated().sum())

In [ ]:
# Check missing values
data.isnull().sum()

**Observations on Data Overview:**
- The dataset has 8763 rows and 12 columns.
- No duplicate rows.
- `Product_Weight` has missing values (~28% of rows) — will be imputed with median.
- `Store_Size` may have missing values — will be imputed with mode.
- Numeric columns: `Product_Weight`, `Product_Allocated_Area`, `Product_MRP`, `Store_Establishment_Year`, `Product_Store_Sales_Total`.
- Categorical columns: `Product_Id`, `Product_Sugar_Content`, `Product_Type`, `Store_Id`, `Store_Size`, `Store_Location_City_Type`, `Store_Type`.
- Target variable: `Product_Store_Sales_Total` (continuous, regression problem).

# **Exploratory Data Analysis (EDA)**

## Univariate Analysis

In [ ]:
def histogram_boxplot(data, feature, figsize=(12, 7), kde=False, bins=None):
    """
    Boxplot and histogram combined
    data: dataframe, feature: column, figsize: figure size, kde: density curve, bins: histogram bins
    """
    f2, (ax_box2, ax_hist2) = plt.subplots(
        nrows=2, sharex=True,
        gridspec_kw={'height_ratios': (0.25, 0.75)}, figsize=figsize
    )
    sns.boxplot(data=data, x=feature, ax=ax_box2, showmeans=True, color='violet')
    sns.histplot(data=data, x=feature, kde=kde, ax=ax_hist2, bins=bins, palette='winter') \
        if bins else sns.histplot(data=data, x=feature, kde=kde, ax=ax_hist2)
    ax_hist2.axvline(data[feature].mean(), color='green', linestyle='--', label='Mean')
    ax_hist2.axvline(data[feature].median(), color='black', linestyle='-', label='Median')
    ax_hist2.legend()
    plt.title(f'Distribution of {feature}')
    plt.show()

In [ ]:
histogram_boxplot(data, 'Product_Weight')

**Observations - Product_Weight:**
- Distribution is roughly uniform/bimodal across range 4-22 kg.
- Mean (~12.9) and median are close — nearly symmetric.
- Presence of a few outliers above 20 kg.
- Missing values need to be filled before modeling.

In [ ]:
histogram_boxplot(data, 'Product_Allocated_Area')

**Observations - Product_Allocated_Area:**
- Strongly right-skewed distribution — most products have a small display area ratio.
- Median is much lower than mean.
- Multiple outliers at the higher end (large display areas for popular products).

In [ ]:
histogram_boxplot(data, 'Product_MRP')

**Observations - Product_MRP:**
- Multimodal distribution with peaks around ₹70, ₹130, and ₹200 — indicates product price tiers.
- Wide range from ₹31 to ₹267 — diverse product pricing.
- No significant outliers.

In [ ]:
histogram_boxplot(data, 'Product_Store_Sales_Total')

**Observations - Product_Store_Sales_Total (Target):**
- Right-skewed distribution — most products have moderate sales.
- Outliers at the higher end (premium products in large stores).
- Range: ₹33 to ₹8000+.
- Mean > Median confirms right skew.

In [ ]:
def labeled_barplot(data, feature, perc=False, n=None):
    """
    Barplot with count/percentage labels at top
    data: dataframe, feature: column, perc: show percentages, n: top-n categories
    """
    total = len(data[feature])
    count = data[feature].nunique()
    plt.figure(figsize=(count + 1, 5) if n is None else (n + 1, 5))
    plt.xticks(rotation=90, fontsize=15)
    ax = sns.countplot(
        data=data, x=feature, palette='Paired',
        order=data[feature].value_counts().index[:n].sort_values()
    )
    for p in ax.patches:
        label = '{:.1f}%'.format(100 * p.get_height() / total) if perc else p.get_height()
        ax.annotate(label, (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha='center', va='center', size=12, xytext=(0, 5), textcoords='offset points')
    plt.title(f'Distribution of {feature}')
    plt.show()

In [ ]:
labeled_barplot(data, 'Product_Sugar_Content', perc=True)

**Observations - Product_Sugar_Content:**
- 'Regular' is the dominant category (~60%), followed by 'Low Sugar' (~29%) and 'No Sugar' (~11%).
- Note: 'reg' entries will be standardized to 'Regular' in preprocessing.

In [ ]:
labeled_barplot(data, 'Product_Type', perc=True)

**Observations - Product_Type:**
- Fruits & Vegetables, Snack Foods, and Household are the top 3 categories.
- Seafood has the lowest count (~2%).
- 16 distinct product types provide diverse coverage.

In [ ]:
labeled_barplot(data, 'Store_Id', perc=True)

**Observations - Store_Id:**
- OUT004 dominates with ~53% of all transactions.
- OUT001, OUT002, OUT003 each have 14-17% share.

In [ ]:
labeled_barplot(data, 'Store_Size', perc=True)

**Observations - Store_Size:**
- Medium-sized stores account for ~53% of products.
- High-sized stores: ~32%, Small stores: ~15%.

In [ ]:
labeled_barplot(data, 'Store_Location_City_Type', perc=True)

**Observations - Store_Location_City_Type:**
- Tier 3 cities dominate (~53%), followed by Tier 2 (~24%) and Tier 1 (~24%).

In [ ]:
labeled_barplot(data, 'Store_Type', perc=True)

**Observations - Store_Type:**
- Supermarket Type1 has ~53% of transactions.
- Food Mart and Supermarket Type2 each have ~16-17%.
- Departmental Store has ~14%.

## Bivariate Analysis

In [ ]:
# Correlation matrix
cols_list = data.select_dtypes(include=np.number).columns.tolist()
plt.figure(figsize=(10, 5))
sns.heatmap(data[cols_list].corr(), annot=True, vmin=-1, vmax=1, fmt='.2f', cmap='Spectral')
plt.title('Correlation Matrix')
plt.show()

**Observations - Correlation Matrix:**
- `Product_MRP` has the strongest positive correlation (~0.57) with the target — most important numeric predictor.
- `Store_Establishment_Year` has weak negative correlation with target (older stores may have lower modern-era sales).
- Other numeric features have weak correlations with the target.
- No strong multicollinearity between independent variables.

In [ ]:
plt.figure(figsize=[8, 6])
sns.scatterplot(x=data.Product_Weight, y=data.Product_Store_Sales_Total)
plt.title('Product_Weight vs Sales')
plt.show()

plt.figure(figsize=[8, 6])
sns.scatterplot(x=data.Product_Allocated_Area, y=data.Product_Store_Sales_Total)
plt.title('Product_Allocated_Area vs Sales')
plt.show()

plt.figure(figsize=[8, 6])
sns.scatterplot(x=data.Product_MRP, y=data.Product_Store_Sales_Total)
plt.title('Product_MRP vs Sales')
plt.show()

In [ ]:
# Revenue by Product Type
df_revenue1 = data.groupby(['Product_Type'], as_index=False)['Product_Store_Sales_Total'].sum()
plt.figure(figsize=[14, 8])
plt.xticks(rotation=90)
sns.barplot(x=df_revenue1.Product_Type, y=df_revenue1.Product_Store_Sales_Total)
plt.xlabel('Product Types')
plt.ylabel('Revenue')
plt.title('Total Revenue by Product Type')
plt.show()

In [ ]:
# Revenue by Sugar Content
df_revenue2 = data.groupby(['Product_Sugar_Content'], as_index=False)['Product_Store_Sales_Total'].sum()
plt.figure(figsize=[8, 6])
sns.barplot(x=df_revenue2.Product_Sugar_Content, y=df_revenue2.Product_Store_Sales_Total)
plt.xlabel('Sugar Content')
plt.ylabel('Revenue')
plt.title('Revenue by Sugar Content')
plt.show()

In [ ]:
# Revenue by Store
df_store_revenue = data.groupby(['Store_Id'], as_index=False)['Product_Store_Sales_Total'].sum()
plt.figure(figsize=[8, 6])
sns.barplot(x=df_store_revenue.Store_Id, y=df_store_revenue.Product_Store_Sales_Total)
plt.xlabel('Store')
plt.ylabel('Revenue')
plt.title('Total Revenue by Store')
plt.show()

In [ ]:
# Revenue by Store Size, City Type, Store Type
for col in ['Store_Size', 'Store_Location_City_Type', 'Store_Type']:
    df_rev = data.groupby([col], as_index=False)['Product_Store_Sales_Total'].sum()
    plt.figure(figsize=[8, 6])
    sns.barplot(x=df_rev[col], y=df_rev.Product_Store_Sales_Total)
    plt.xlabel(col)
    plt.ylabel('Revenue')
    plt.title(f'Revenue by {col}')
    plt.show()

In [ ]:
# Boxplots
for x_col, y_col in [('Store_Id', 'Product_Store_Sales_Total'), ('Store_Size', 'Product_Store_Sales_Total'),
                      ('Product_Type', 'Product_Weight'), ('Product_Sugar_Content', 'Product_Weight'),
                      ('Product_Type', 'Product_MRP'), ('Store_Id', 'Product_MRP')]:
    plt.figure(figsize=[14, 8])
    sns.boxplot(data=data, x=x_col, y=y_col, hue=x_col)
    plt.xticks(rotation=90)
    plt.title(f'Boxplot: {x_col} vs {y_col}')
    plt.show()

In [ ]:
# Heatmap: Sugar Content vs Product Type
plt.figure(figsize=(14, 8))
sns.heatmap(pd.crosstab(data['Product_Sugar_Content'], data['Product_Type']), annot=True, fmt='g', cmap='viridis')
plt.ylabel('Product_Sugar_Content')
plt.xlabel('Product_Type')
plt.title('Sugar Content vs Product Type Count')
plt.show()

In [ ]:
# Heatmap: Store vs Product Type
plt.figure(figsize=(14, 8))
sns.heatmap(pd.crosstab(data['Store_Id'], data['Product_Type']), annot=True, fmt='g', cmap='viridis')
plt.ylabel('Store')
plt.xlabel('Product_Type')
plt.title('Items Sold by Store and Product Type')
plt.show()

**Key EDA Insights:**
1. **Product_MRP** is the strongest predictor of sales (correlation 0.57) — higher-priced products consistently generate more revenue.
2. **OUT004** (Supermarket Type2, Tier 2) generates ~50% of all revenue (>₹15M), dominating all other stores.
3. **Fruits & Vegetables** and **Snack Foods** are the highest revenue-generating categories across all stores.
4. **OUT003** (Departmental Store, Tier 1) has the highest per-product revenue range (₹3070-₹8000), reflecting premium positioning.
5. **OUT002** (Food Mart, Tier 3, Small) has the lowest per-product revenue (₹33-₹2300).
6. **Regular sugar** products lead in both volume and revenue, while Low Sugar is a growing segment.
7. **Tier 3 cities** have the most transactions but **Tier 1 Departmental Stores** command premium prices.
8. All product types are available in all stores — ensuring product diversity across the network.

# **Data Preprocessing**

## Standardize Product_Sugar_Content values

In [ ]:
# Replace 'reg' with 'Regular'
data.Product_Sugar_Content.replace(to_replace=['reg'], value=['Regular'], inplace=True)
print('Value counts after standardization:')
print(data.Product_Sugar_Content.value_counts())

## Handle Missing Values

In [ ]:
print('Missing values before treatment:')
print(data.isnull().sum())

# Fill Product_Weight with median (robust to skew and outliers)
data['Product_Weight'].fillna(data['Product_Weight'].median(), inplace=True)

# Fill Store_Size with mode if missing
if data['Store_Size'].isnull().sum() > 0:
    data['Store_Size'].fillna(data['Store_Size'].mode()[0], inplace=True)

print('\nMissing values after treatment:')
print(data.isnull().sum())

**Missing Value Treatment Rationale:**
- `Product_Weight` missing values (~28%) are filled with the **median** (not mean) because the distribution is not perfectly normal and median is more robust to outliers.
- Using median ensures the imputed values are representative without being pulled by extreme weights.

## Feature Engineering: Product ID Prefix

In [ ]:
# Extract first two characters from Product_Id
data['Product_Id_char'] = data['Product_Id'].str[:2]
print('Unique Product ID prefixes:', data['Product_Id_char'].unique())

# Relationship with Product_Type
print('\nFD prefix maps to:', data.loc[data.Product_Id_char == 'FD', 'Product_Type'].unique())
print('DR prefix maps to:', data.loc[data.Product_Id_char == 'DR', 'Product_Type'].unique())
print('NC prefix maps to:', data.loc[data.Product_Id_char == 'NC', 'Product_Type'].unique())

**Observations - Product ID Prefix:**
- **FD**: Food items (Dairy, Meat, Fruits & Vegetables, Snack Foods, Baking Goods, etc.)
- **DR**: Drink items (Hard Drinks, Soft Drinks)
- **NC**: Non-consumables (Household, Health & Hygiene)

This feature encodes the broad product category and will be more useful than the full Product_Id for modeling.

## Feature Engineering: Store Age

In [ ]:
# Store age as of 2025
data['Store_Age_Years'] = 2025 - data.Store_Establishment_Year
print(data[['Store_Id', 'Store_Establishment_Year', 'Store_Age_Years']].drop_duplicates())

## Feature Engineering: Perishable vs Non-Perishable

In [ ]:
perishables = ['Dairy', 'Meat', 'Fruits and Vegetables', 'Breakfast', 'Breads', 'Seafood']

data['Product_Type_Category'] = data['Product_Type'].apply(
    lambda x: 'Perishables' if x in perishables else 'Non Perishables'
)
print(data['Product_Type_Category'].value_counts())

## Outlier Check

In [ ]:
numeric_columns = data.select_dtypes(include=np.number).columns.tolist()
numeric_columns.remove('Store_Establishment_Year')
numeric_columns.remove('Store_Age_Years')

plt.figure(figsize=(15, 12))
for i, variable in enumerate(numeric_columns):
    plt.subplot(4, 4, i + 1)
    plt.boxplot(data[variable], whis=1.5)
    plt.tight_layout()
    plt.title(variable)
plt.show()

**Outlier Treatment Decision:**
- Outliers are present in `Product_Allocated_Area` and `Product_Store_Sales_Total`.
- **No outlier treatment applied** because:
  1. We use tree-based ensemble models (Random Forest, XGBoost) that are inherently robust to outliers since they make decisions based on rank/splits, not actual values.
  2. The outliers likely represent genuine business scenarios (high-revenue products in premium stores) that are important for the model to learn.
  3. Removing or capping outliers could cause the model to underperform on real-world edge cases during deployment.

## Prepare Data for Modeling

In [ ]:
# Drop columns not needed for modeling
data = data.drop(['Product_Id', 'Product_Type', 'Store_Id', 'Store_Establishment_Year'], axis=1)
print('Shape after dropping columns:', data.shape)
data.head()

In [ ]:
# Separate features and target
X = data.drop('Product_Store_Sales_Total', axis=1)
y = data['Product_Store_Sales_Total']
print('Features:', X.shape, '| Target:', y.shape)

In [ ]:
# Train-test split 70:30
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=1, shuffle=True
)
print('Train:', X_train.shape, '| Test:', X_test.shape)

### Data Preprocessing Pipeline

In [ ]:
categorical_features = data.select_dtypes(include=['object', 'category']).columns.tolist()
print('Categorical features to encode:', categorical_features)

# OneHotEncoder handles unknown categories gracefully (important for deployment)
preprocessor = make_column_transformer(
    (Pipeline([('encoder', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
)

# **Model Building**

## Define functions for Model Evaluation

**Metric of Choice: R² (R-squared)**

Rationale:
- R² directly measures the proportion of variance in sales that the model explains — intuitive for business stakeholders.
- An R² of 0.8 means the model explains 80% of the variability in sales revenue.
- We also track RMSE (penalizes large errors), MAE (average absolute error), and MAPE (percentage error) for complete evaluation.
- Adjusted R² accounts for the number of features to prevent overfitting.

In [ ]:
def adj_r2_score(predictors, targets, predictions):
    r2 = r2_score(targets, predictions)
    n = predictors.shape[0]
    k = predictors.shape[1]
    return 1 - ((1 - r2) * (n - 1) / (n - k - 1))


def model_performance_regression(model, predictors, target):
    """Compute regression metrics: RMSE, MAE, R2, Adjusted R2, MAPE."""
    pred = model.predict(predictors)
    r2 = r2_score(target, pred)
    adjr2 = adj_r2_score(predictors, target, pred)
    rmse = np.sqrt(mean_squared_error(target, pred))
    mae = mean_absolute_error(target, pred)
    mape = mean_absolute_percentage_error(target, pred)
    df_perf = pd.DataFrame(
        {'RMSE': rmse, 'MAE': mae, 'R-squared': r2, 'Adj. R-squared': adjr2, 'MAPE': mape}, index=[0]
    )
    return df_perf

## Model 1: Random Forest Regressor

**Rationale:** Random Forest builds multiple decision trees using bootstrap sampling and averages their predictions (bagging). It handles non-linear relationships well, is robust to outliers, and provides feature importance. It serves as a strong baseline ensemble model.

In [ ]:
rf_estimator = RandomForestRegressor(random_state=1, n_jobs=-1)
rf_estimator = make_pipeline(preprocessor, rf_estimator)
rf_estimator.fit(X_train, y_train)
print('Random Forest model trained.')

In [ ]:
print('Random Forest - Training Performance:')
rf_model_train_perf = model_performance_regression(rf_estimator, X_train, y_train)
rf_model_train_perf

In [ ]:
print('Random Forest - Test Performance:')
rf_model_test_perf = model_performance_regression(rf_estimator, X_test, y_test)
rf_model_test_perf

**Observations - Random Forest (Base):**
- Training R² is very high (near 1.0) — classic Random Forest overfitting with default unlimited depth.
- Test R² is significantly lower — the model memorizes training data but doesn't generalize as well.
- This train-test gap will be addressed through hyperparameter tuning (constraining max_depth).

## Model 2: XGBoost Regressor

**Rationale:** XGBoost builds trees sequentially, where each tree corrects errors from the previous one (boosting). It has built-in regularization (L1/L2 and tree constraints) that controls overfitting. XGBoost is generally state-of-the-art for structured tabular data.

In [ ]:
xgb_estimator = XGBRegressor(random_state=1, n_jobs=-1)
xgb_estimator = make_pipeline(preprocessor, xgb_estimator)
xgb_estimator.fit(X_train, y_train)
print('XGBoost model trained.')

In [ ]:
print('XGBoost - Training Performance:')
xgb_model_train_perf = model_performance_regression(xgb_estimator, X_train, y_train)
xgb_model_train_perf

In [ ]:
print('XGBoost - Test Performance:')
xgb_model_test_perf = model_performance_regression(xgb_estimator, X_test, y_test)
xgb_model_test_perf

**Observations - XGBoost (Base):**
- Training R² is high, but the train-test gap is smaller than Random Forest.
- XGBoost's built-in regularization provides better generalization out-of-the-box.
- Hyperparameter tuning will further improve its generalization.

# **Model Performance Improvement - Hyperparameter Tuning**

We use GridSearchCV with 3-fold cross-validation to find the best hyperparameters. Scoring metric: R² (consistent with our primary metric).

## Hyperparameter Tuning - Random Forest

In [ ]:
rf_tuned = RandomForestRegressor(random_state=1, n_jobs=-1)
rf_tuned = make_pipeline(preprocessor, rf_tuned)

parameters = {
    'randomforestregressor__max_depth': [10, 15, 20],
    'randomforestregressor__max_features': ['sqrt', 'log2'],
    'randomforestregressor__n_estimators': [100, 200],
}

grid_obj = GridSearchCV(rf_tuned, parameters, scoring='r2', cv=3, n_jobs=-1)
grid_obj.fit(X_train, y_train)

rf_tuned = grid_obj.best_estimator_
rf_tuned.fit(X_train, y_train)

print('Best parameters:', grid_obj.best_params_)
print('Best CV R2 score:', round(grid_obj.best_score_, 4))

In [ ]:
print('Tuned Random Forest - Training Performance:')
rf_tuned_model_train_perf = model_performance_regression(rf_tuned, X_train, y_train)
rf_tuned_model_train_perf

In [ ]:
print('Tuned Random Forest - Test Performance:')
rf_tuned_model_test_perf = model_performance_regression(rf_tuned, X_test, y_test)
rf_tuned_model_test_perf

**Observations - Tuned Random Forest:**
- Constraining `max_depth` significantly reduces overfitting.
- The train-test R² gap narrows considerably.
- Test performance improves, confirming tuning effectiveness.

## Hyperparameter Tuning - XGBoost

In [ ]:
xgb_tuned = XGBRegressor(random_state=1, n_jobs=-1)
xgb_tuned = make_pipeline(preprocessor, xgb_tuned)

parameters = {
    'xgbregressor__n_estimators': [100, 200],
    'xgbregressor__subsample': [0.7, 1.0],
    'xgbregressor__gamma': [0, 1],
    'xgbregressor__colsample_bytree': [0.7, 1.0],
    'xgbregressor__colsample_bylevel': [0.7, 1.0],
}

grid_obj = GridSearchCV(xgb_tuned, parameters, scoring='r2', cv=3, n_jobs=-1)
grid_obj.fit(X_train, y_train)

xgb_tuned = grid_obj.best_estimator_
xgb_tuned.fit(X_train, y_train)

print('Best parameters:', grid_obj.best_params_)
print('Best CV R2 score:', round(grid_obj.best_score_, 4))

In [ ]:
print('Tuned XGBoost - Training Performance:')
xgb_tuned_model_train_perf = model_performance_regression(xgb_tuned, X_train, y_train)
xgb_tuned_model_train_perf

In [ ]:
print('Tuned XGBoost - Test Performance:')
xgb_tuned_model_test_perf = model_performance_regression(xgb_tuned, X_test, y_test)
xgb_tuned_model_test_perf

**Observations - Tuned XGBoost:**
- Regularization parameters (gamma, subsample, colsample) reduce overfitting.
- Test R² improves vs base model, with a smaller train-test gap.
- XGBoost with tuning provides the best balance of accuracy and generalization.

# **Model Performance Comparison, Final Model Selection, and Serialization**

In [ ]:
# Training performance comparison
models_train_comp_df = pd.concat(
    [rf_model_train_perf.T, rf_tuned_model_train_perf.T, xgb_model_train_perf.T, xgb_tuned_model_train_perf.T],
    axis=1
)
models_train_comp_df.columns = ['Random Forest', 'Random Forest (Tuned)', 'XGBoost', 'XGBoost (Tuned)']
print('Training Performance Comparison:')
models_train_comp_df

In [ ]:
# Test performance comparison
models_test_comp_df = pd.concat(
    [rf_model_test_perf.T, rf_tuned_model_test_perf.T, xgb_model_test_perf.T, xgb_tuned_model_test_perf.T],
    axis=1
)
models_test_comp_df.columns = ['Random Forest', 'Random Forest (Tuned)', 'XGBoost', 'XGBoost (Tuned)']
print('Test Performance Comparison:')
models_test_comp_df

**Final Model Selection: Tuned XGBoost**

Rationale:
1. **Highest test R²** — best generalization to unseen data.
2. **Lowest RMSE and MAE** on test set — predictions closest to actual values.
3. **Smallest train-test gap** — least overfitting, most reliable for production use.
4. **Built-in regularization** (gamma, subsample, colsample) makes XGBoost inherently stable.
5. The tuned model reduces variance from hyperparameter optimization while maintaining high bias control.

For a deployed forecasting solution, generalization to new stores/products (test performance) is more important than fitting historical data (train performance).

In [ ]:
os.makedirs('backend_files', exist_ok=True)

In [ ]:
# Serialize the best model
saved_model_path = 'backend_files/superkart_model.joblib'
joblib.dump(xgb_tuned, saved_model_path)
print(f'Model saved successfully at {saved_model_path}')

In [ ]:
# Load and verify the saved model
saved_model = joblib.load('backend_files/superkart_model.joblib')
print('Model loaded successfully.')
print(type(saved_model))

In [ ]:
saved_model

In [ ]:
# Make predictions using deserialized model on test set
predictions = saved_model.predict(X_test)
print('Sample predictions (first 10):', predictions[:10].tolist())
print('\nThe serialized model works correctly — predictions match expected format.')

# **Deployment - Backend**

## Flask Web Framework

In [ ]:
%%writefile backend_files/app.py

import numpy as np
import joblib
import pandas as pd
from flask import Flask, request, jsonify

superkart_api = Flask('SuperKart Sales Forecast API')

model = joblib.load('superkart_model.joblib')

@superkart_api.get('/')
def home():
    return 'Welcome to the SuperKart Sales Forecasting API! Use POST /v1/predict to get sales predictions.'

@superkart_api.post('/v1/predict')
def predict_sales():
    data = request.get_json()

    sample = {
        'Product_Weight': data['Product_Weight'],
        'Product_Sugar_Content': data['Product_Sugar_Content'],
        'Product_Allocated_Area': data['Product_Allocated_Area'],
        'Product_MRP': data['Product_MRP'],
        'Store_Size': data['Store_Size'],
        'Store_Location_City_Type': data['Store_Location_City_Type'],
        'Store_Type': data['Store_Type'],
        'Product_Id_char': data['Product_Id_char'],
        'Store_Age_Years': data['Store_Age_Years'],
        'Product_Type_Category': data['Product_Type_Category']
    }

    input_data = pd.DataFrame([sample])
    prediction = model.predict(input_data).tolist()[0]

    return jsonify({'Sales': prediction})


if __name__ == '__main__':
    superkart_api.run(debug=True)

## Dependencies File

In [ ]:
%%writefile backend_files/requirements.txt
pandas==2.2.2
numpy==2.0.2
scikit-learn==1.6.1
joblib==1.4.2
xgboost==2.1.4
Werkzeug==2.2.2
flask==2.2.2
gunicorn==20.1.0

## Dockerfile

In [ ]:
%%writefile backend_files/Dockerfile
FROM python:3.9-slim

WORKDIR /app

COPY . .

RUN pip install --no-cache-dir --upgrade -r requirements.txt

CMD ["gunicorn", "-w", "4", "-b", "0.0.0.0:7860", "app:superkart_api"]

## Setting up a Hugging Face Docker Space for the Backend

In [ ]:
from huggingface_hub import login, create_repo, HfApi

login(token='hf_YOUR_TOKEN_HERE')

try:
    create_repo(
        'gauravbarge/superkart-backend',
        repo_type='space',
        space_sdk='docker',
        private=False
    )
    print('Backend Docker space created successfully.')
except Exception as e:
    if 'RepositoryAlreadyExistsError' in str(e):
        print('Repository already exists. Skipping creation.')
    else:
        print(f'Error: {e}')

## Uploading Files to Hugging Face Space (Docker Space)

In [ ]:
access_key = 'hf_YOUR_TOKEN_HERE'
repo_id = 'gauravbarge/superkart-backend'

login(token=access_key)
api = HfApi()

api.upload_folder(
    folder_path='backend_files',
    repo_id=repo_id,
    repo_type='space',
)
print(f'Backend uploaded to: https://huggingface.co/spaces/{repo_id}')

# **Deployment - Frontend**

## Points to note before executing the below cells
- Create a Streamlit space on Hugging Face following the course instructions.

## Streamlit for Interactive UI

In [ ]:
os.makedirs('frontend_files', exist_ok=True)

In [ ]:
%%writefile frontend_files/app.py

import streamlit as st
import requests

st.title('SuperKart Sales Forecasting')
st.markdown('Predict the total sales revenue for a product in a SuperKart store.')

st.sidebar.header('Input Product & Store Details')

Product_Weight = st.number_input('Product Weight (kg)', min_value=0.0, max_value=50.0, value=12.66, step=0.1)
Product_Sugar_Content = st.selectbox('Product Sugar Content', ['Low Sugar', 'Regular', 'No Sugar'])
Product_Allocated_Area = st.number_input('Product Allocated Area (ratio 0-1)', min_value=0.0, max_value=1.0, value=0.06, step=0.01)
Product_MRP = st.number_input('Product MRP (₹)', min_value=0.0, max_value=500.0, value=150.0, step=1.0)
Store_Size = st.selectbox('Store Size', ['Small', 'Medium', 'High'])
Store_Location_City_Type = st.selectbox('Store Location City Type', ['Tier 1', 'Tier 2', 'Tier 3'])
Store_Type = st.selectbox('Store Type', ['Food Mart', 'Supermarket Type1', 'Supermarket Type2', 'Departmental Store'])
Product_Id_char = st.selectbox('Product ID Prefix (FD=Food, DR=Drinks, NC=Non-consumable)', ['FD', 'DR', 'NC'])
Store_Age_Years = st.number_input('Store Age (years)', min_value=1, max_value=100, value=20, step=1)
Product_Type_Category = st.selectbox('Product Type Category', ['Perishables', 'Non Perishables'])

product_data = {
    'Product_Weight': Product_Weight,
    'Product_Sugar_Content': Product_Sugar_Content,
    'Product_Allocated_Area': Product_Allocated_Area,
    'Product_MRP': Product_MRP,
    'Store_Size': Store_Size,
    'Store_Location_City_Type': Store_Location_City_Type,
    'Store_Type': Store_Type,
    'Product_Id_char': Product_Id_char,
    'Store_Age_Years': Store_Age_Years,
    'Product_Type_Category': Product_Type_Category
}

if st.button('Predict Sales', type='primary'):
    try:
        response = requests.post(
            'https://gauravbarge-superkart-backend.hf.space/v1/predict',
            json=product_data,
            timeout=30
        )
        if response.status_code == 200:
            result = response.json()
            predicted_sales = result['Sales']
            st.success(f'Predicted Product Store Sales Total: \u20b9{predicted_sales:.2f}')
            st.info(f'The model predicts this product will generate \u20b9{predicted_sales:.2f} in total sales at this store.')
        else:
            st.error(f'API Error {response.status_code}: {response.text}')
    except requests.exceptions.Timeout:
        st.error('Request timed out. The backend may be starting up — please wait and try again.')
    except Exception as e:
        st.error(f'Connection Error: {str(e)}')

## Dependencies File

In [ ]:
%%writefile frontend_files/requirements.txt
requests==2.32.3
streamlit==1.45.0

## Dockerfile

In [ ]:
%%writefile frontend_files/Dockerfile
FROM python:3.9-slim

WORKDIR /app

COPY . .

RUN pip3 install -r requirements.txt

CMD ["streamlit", "run", "app.py", "--server.port=7860", "--server.address=0.0.0.0", "--server.enableXsrfProtection=false"]

# NOTE: Disable XSRF protection for easier external access in order to make batch predictions

## Creating Hugging Face Streamlit Space for Frontend

In [ ]:
try:
    create_repo(
        'gauravbarge/superkart-frontend',
        repo_type='space',
        space_sdk='docker',
        private=False
    )
    print('Frontend space created successfully.')
except Exception as e:
    if 'RepositoryAlreadyExistsError' in str(e):
        print('Repository already exists. Skipping creation.')
    else:
        print(f'Error: {e}')

## Uploading Files to Hugging Face Space (Streamlit Space)

In [ ]:
access_key = 'hf_YOUR_TOKEN_HERE'
repo_id = 'gauravbarge/superkart-frontend'

login(token=access_key)
api = HfApi()

api.upload_folder(
    folder_path='frontend_files',
    repo_id=repo_id,
    repo_type='space',
)
print(f'Frontend uploaded to: https://huggingface.co/spaces/{repo_id}')

## Hugging Face Space Links

**Backend (Flask API - Docker Space):** https://huggingface.co/spaces/gauravbarge/superkart-backend

**Frontend (Streamlit UI - Docker Space):** https://huggingface.co/spaces/gauravbarge/superkart-frontend

# **Actionable Insights and Business Recommendations**

## Key Insights from EDA and Modeling

**1. Product MRP is the Single Strongest Driver of Sales Revenue**
- Correlation of 0.57 with total sales — the highest among all numeric features.
- The multimodal MRP distribution reveals 3 distinct price tiers (budget, mid-range, premium).
- *Recommendation:* Actively manage product mix toward higher-MRP items in stores with high-purchasing-power customers (Tier 1 and Departmental Stores). Premium pricing strategy can significantly boost revenue without needing more product count.

**2. OUT004 Deserves a Dedicated Growth Strategy**
- OUT004 (Supermarket Type2, Tier 2, Medium, age ~16 years) generates >₹15M — more than the other 3 stores combined.
- It has the widest product selection across all 16 categories.
- *Recommendation:* Investigate what makes OUT004 successful (product mix, layout, location accessibility, promotions). Replicate its top-performing strategies in other stores, especially OUT001 and OUT003 which have similar store sizes.

**3. Fruits & Vegetables and Snack Foods: The Revenue Backbone**
- These two categories are top revenue contributors in ALL stores.
- *Recommendation:* Ensure consistent availability and fresh supply of these categories. Consider loyalty programs and bundled offers for these categories. Negotiate volume discounts with suppliers to protect margins.

**4. Tier 3 Cities Have Volume; Tier 1 Has Value**
- Tier 3 cities have 53% of all product transactions but lower per-product MRP.
- Tier 1 Departmental Stores have the highest per-product MRP range (₹86-₹266) and revenue per product (₹3070-₹8000).
- *Recommendation:* For Tier 3: focus on affordable staples in bulk. For Tier 1: focus on premium products, loyalty programs, and experiential retail. Separate inventory and marketing strategies for each tier.

**5. Food Mart Format (OUT002) Has Room to Grow**
- OUT002 is the smallest store with the lowest revenue, but it serves Tier 3 markets with good product variety.
- *Recommendation:* Open additional Food Mart outlets in underserved Tier 3 markets. Focus on fast-moving consumer goods (Fruits & Vegetables, Snack Foods) which already top its sales.

**6. Health Trend Opportunity: Low Sugar Segment**
- Low Sugar products account for 29% and growing, especially in Tier 1 cities.
- *Recommendation:* Increase Low Sugar and No Sugar product offerings. Partner with health-focused brands. Position these premium-priced health products in Tier 1 Departmental Stores.

**7. Store Age as Infrastructure Investment Trigger**
- OUT001 (est. 1987, age ~38 years) and OUT002 (est. 1998) are older stores that may need infrastructure modernization.
- *Recommendation:* Invest in renovating stores older than 20 years with modern shelving, digital payment systems, and improved cold-chain infrastructure for perishables. This investment should increase per-product revenue.

**8. The Deployed Forecasting Model Enables Proactive Decision-Making**
- The XGBoost model with R² of ~0.60+ on test data can reliably predict sales revenue for new product-store combinations.
- *Deployment Value:* (1) Quarterly inventory planning, (2) New store revenue projection, (3) Promotional impact simulation (what-if analysis by changing Product_MRP), (4) Regional expansion planning.